**Requirements:**

The `R` package `magick` (which depends on `Rcpp`) is required to process the images (eg. to extract RGB channels). The following `R` packages were installed as follows on JupyterLab:

In [1]:
library(future)
library(future.apply)
library(progressr)
library(magick)
packageVersion("magick")

source("./fn-img-tools.R")

Linking to ImageMagick 7.1.2.26
Enabled features: cairo, fontconfig, freetype, fftw, heic, lcms, raw, rsvg, webp, x11
Disabled features: ghostscript, pango



[1] ‘2.9.1’

In [3]:
unzip("./archive_weather.zip", exdir = "original-img-weather-dataset")

In [4]:
available_workers <- parallel::detectCores()
nworkers <- 50 #parallel::detectCores() - 1
cat(paste(sprintf("Using %d cores out of %d available cores.", nworkers, available_workers)))

Using 50 cores out of 256 available cores.

In [5]:
input_dir <- "original-img-weather-dataset"
output_dir <- "img-weather-thumbs-150x100"

target_width <- 150L
target_height <- 100L

img_paths <- list.files(path = input_dir, pattern = "\\.(jpg|jpeg|png|gif)$",
    recursive = TRUE, full.names = TRUE, ignore.case = TRUE)

img_paths <- normalizePath(img_paths, mustWork = TRUE)

dir.create(output_dir)

In [9]:
img_path <- img_paths[1]
  rel_path <- sub(
    paste0("^", normalizePath(input_dir, mustWork = TRUE), .Platform$file.sep),
    "", normalizePath(img_path, mustWork = TRUE))

  out_path <- file.path(output_dir,
    paste0(tools::file_path_sans_ext(rel_path), ".png"))
out_path
sub("/dataset/", "/", out_path,fixed = TRUE)

[1] "img-weather-thumbs-150x100/dataset/dew/2208.png"

[1] "img-weather-thumbs-150x100/dew/2208.png"

In [7]:
future::plan( future::multisession, workers = nworkers)

progressr::handlers("txtprogressbar")

niter <- length(img_paths)

res_list <- progressr::with_progress({
  p <- progressr::progressor(along = seq_len(niter))

  future.apply::future_lapply(
    X = seq_len(niter),
    FUN = function(i) {
      res <- resize_and_fill(img_paths[[i]], 
        "original-img-weather-dataset", "img-weather-thumbs-150x100", 
        150, 100, "dataset")
      p() # update progress
      res
    },
    future.seed = NULL
  )
})

future::plan(future::sequential) # back to sequential processing

In [11]:
resize_log_df <- do.call(
  rbind,
  lapply(res_list, function(x) {
    data.frame(ok = x$ok, input = x$input, output = x$output,
      msg = x$msg, stringsAsFactors = FALSE)
  })
)
cat("Successful:", sum(resize_log_df$ok), "\n")
cat("Failed:", sum(!resize_log_df$ok), "\n")

Successful: 6862 
Failed: 0 


In [8]:
# remove the images directory to save space.
# To see the images unzip archive_weather.zip and browse it or 
# browse img-weather-thumbs-150x100, containing the smaller thumbnails created in this Notebook.
unlink("./original-img-weather-dataset", recursive = TRUE, force = TRUE)